# Weekly Stat Arb Model — 5 Buckets, Long/Short 10%
Strategy: every week long top 10% (Strong Buy) and short bottom 10% (Strong Avoid).
5 buckets: Strong Avoid / Avoid / Neutral / Buy / Strong Buy — split at 10/30/70/90 percentiles.

In [ ]:
from __future__ import annotations
from pathlib import Path
import warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
warnings.filterwarnings('ignore')

IN_FILE = Path('Models/Data/data_for_experiments.csv')
OUT_DIR = Path('Models/Figures')

TRAIN_CUTOFF = '2024-01'
FORWARD_DAYS = 7
MIN_COINS    = 20
TOP_N        = 30

# 5 buckets split at 10 / 30 / 70 / 90 percentiles
LABEL_ORDER = ['Strong Avoid', 'Avoid', 'Neutral', 'Buy', 'Strong Buy']
LABEL_MAP   = {'Strong Avoid': 0, 'Avoid': 1, 'Neutral': 2, 'Buy': 3, 'Strong Buy': 4}
BINS        = [0, 0.10, 0.30, 0.70, 0.90, 1.001]
COLORS      = {
    'Strong Buy':   '#1D9E75',
    'Buy':          '#97C459',
    'Neutral':      '#888780',
    'Avoid':        '#F0997B',
    'Strong Avoid': '#E24B4A',
}
BTC_COLOR = '#F7931A'
ETH_COLOR = '#627EEA'

# Strategy: long Strong Buy, short Strong Avoid
LONG_LABEL  = 'Strong Buy'
SHORT_LABEL = 'Strong Avoid'

LEAKY_COLS = {
    'label', 'forward_return_7d', 'forward_sharpe_7d', 'forward_sharpe_rank',
    'return_1d_rank', 'return_1d_zscore', 'return_7d_rank', 'return_7d_zscore',
    'return_30d_rank', 'return_30d_zscore', 'volatility_30d_rank', 'volatility_30d_zscore',
    'rsi_14_rank', 'rsi_14_zscore', 'macd_hist_rank', 'macd_hist_zscore',
    'bb_pct_rank', 'bb_pct_zscore', 'atr_pct_rank', 'atr_pct_zscore',
    'obv_divergence_rank', 'obv_divergence_zscore', 'stoch_k_rank', 'stoch_k_zscore',
    'adx_rank', 'adx_zscore', 'volume_vs_30d_avg_rank', 'volume_vs_30d_avg_zscore',
    'drawdown_from_90d_peak_rank', 'drawdown_from_90d_peak_zscore',
    'price_vs_ath_rank', 'price_vs_ath_zscore', 'range_position_30d_rank',
    'range_position_30d_zscore', 'consecutive_up_days_rank', 'consecutive_up_days_zscore',
    'consecutive_down_days_rank', 'consecutive_down_days_zscore',
    'coin_age_days_rank', 'coin_age_days_zscore',
    'momentum_score', 'mean_reversion_score', 'trend_score',
    'asset_id', 'year_week', 'date', 'timestamp', 'exchange',
    'pair_symbol', 'source', 'open', 'high', 'low', 'close',
    'granularity', 'is_active',
}

CS_FEATURES = [
    'return_1d', 'return_7d', 'return_30d', 'volatility_30d',
    'rsi_14', 'macd_hist', 'bb_pct', 'atr_pct', 'obv_divergence',
    'stoch_k', 'adx', 'volume_vs_30d_avg', 'drawdown_from_90d_peak',
    'price_vs_ath', 'range_position_30d', 'consecutive_up_days',
    'consecutive_down_days', 'coin_age_days', 'galaxy_score', 'alt_rank',
    'market_cap_usd', 'coin_mcap_share_recalc', 'oi_usd', 'funding_rate',
    'taker_buy_ratio',
]
print('Config loaded. Buckets:', LABEL_ORDER)

## 1. Load & build weekly labels (5 buckets)

In [ ]:
df = pd.read_csv(IN_FILE, low_memory=False)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['asset_id', 'date']).reset_index(drop=True)
df['year_week'] = df['date'].dt.strftime('%G-W%V')

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Coins : {df["asset_id"].nunique()}')
print(f'Range : {df["date"].min().date()} -> {df["date"].max().date()}')

def compute_forward_sharpe_7d(group):
    closes  = group['close'].values
    results = []
    for i in range(len(closes)):
        end = i + FORWARD_DAYS
        if end >= len(closes):
            results.append(np.nan)
            continue
        window     = closes[i:end]
        daily_rets = np.diff(window) / window[:-1]
        fwd_ret    = (window[-1] - window[0]) / window[0]
        vol        = daily_rets.std() * np.sqrt(365)
        results.append((fwd_ret / vol) if vol > 1e-8 else 0.0)
    return pd.Series(results, index=group.index)

print('Computing forward Sharpe (7d)...')
df['forward_sharpe_7d'] = (
    df.groupby('asset_id', group_keys=False)
    .apply(compute_forward_sharpe_7d)
)
df['forward_return_7d'] = (
    df.groupby('asset_id')['close']
    .transform(lambda x: x.shift(-FORWARD_DAYS) / x - 1)
)

weekly = (
    df.dropna(subset=['forward_sharpe_7d'])
    .sort_values('date')
    .groupby(['asset_id', 'year_week'])
    .last()
    .reset_index()
)

def assign_labels(group):
    if len(group) < MIN_COINS:
        return group.assign(label=np.nan, forward_sharpe_rank=np.nan)
    q = group['forward_sharpe_7d'].rank(pct=True)
    group = group.copy()
    group['forward_sharpe_rank'] = q
    group['label'] = pd.cut(
        q, bins=BINS,
        labels=LABEL_ORDER,
        include_lowest=True,
    )
    return group

weekly = (
    weekly.groupby('year_week', group_keys=False)
    .apply(assign_labels)
    .dropna(subset=['label'])
    .reset_index(drop=True)
)

print(f'Weekly snapshots: {len(weekly):,} rows, {weekly["year_week"].nunique()} weeks')
print('Label distribution:')
print(weekly['label'].value_counts()[LABEL_ORDER].to_string())

## 2. Cross-sectional features & two-pass RF

In [ ]:
cs_cols = [c for c in CS_FEATURES if c in weekly.columns]
for col in cs_cols:
    weekly[f'{col}_rank']   = weekly.groupby('year_week')[col].transform(lambda x: x.rank(pct=True))
    weekly[f'{col}_zscore'] = weekly.groupby('year_week')[col].transform(
        lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1)
    )
print(f'Cross-sectional features: {len(cs_cols)*2} new columns')

cutoff_date = pd.Timestamp(TRAIN_CUTOFF + '-01')
train_df = weekly[pd.to_datetime(weekly['date']) <  cutoff_date].copy()
test_df  = weekly[pd.to_datetime(weekly['date']) >= cutoff_date].copy()
print(f'Train: {len(train_df):,} rows ({train_df["year_week"].min()} -> {train_df["year_week"].max()})')
print(f'Test : {len(test_df):,} rows ({test_df["year_week"].min()} -> {test_df["year_week"].max()})')

all_features = [
    c for c in weekly.columns
    if c not in LEAKY_COLS
    and weekly[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]
null_rates   = train_df[all_features].isnull().mean()
all_features = [f for f in all_features if null_rates[f] <= 0.30]

def prep(d, cols):
    return d[cols].fillna(0).replace([np.inf, -np.inf], 0).clip(-1e9, 1e9)

y_train = train_df['label'].map(LABEL_MAP)
y_test  = test_df['label'].map(LABEL_MAP)

print(f'Pass 1: {len(all_features)} features...')
rf1 = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5,
                              max_features='sqrt', class_weight='balanced',
                              oob_score=True, n_jobs=-1, random_state=42)
rf1.fit(prep(train_df, all_features), y_train)
imp1  = pd.Series(rf1.feature_importances_, index=all_features).sort_values(ascending=False)
top30 = imp1.head(TOP_N).index.tolist()
print(f'Pass 1 OOB: {rf1.oob_score_*100:.2f}%')

print(f'Pass 2: top {TOP_N} features...')
model = RandomForestClassifier(n_estimators=500, max_depth=12, min_samples_leaf=5,
                                max_features='sqrt', class_weight='balanced',
                                oob_score=True, n_jobs=-1, random_state=42)
model.fit(prep(train_df, top30), y_train)
y_pred  = model.predict(prep(test_df, top30))
acc     = accuracy_score(y_test, y_pred)
baseline = 1 / len(LABEL_ORDER)

print(f'Pass 1 OOB ({len(all_features)} feat): {rf1.oob_score_*100:.2f}%')
print(f'Pass 2 OOB ({TOP_N} feat)           : {model.oob_score_*100:.2f}%')
print(f'Test accuracy: {acc*100:.2f}%  (vs {baseline*100:.0f}% baseline)')
print(classification_report(y_test, y_pred, target_names=LABEL_ORDER))

test_df = test_df.copy()
test_df['predicted'] = [LABEL_ORDER[p] for p in y_pred]

with open(MODEL_FILE, 'wb') as f:
    pickle.dump(model, f)
print(f'Model saved -> {MODEL_FILE.name}')

## 3. Strategy P&L — long Strong Buy, short Strong Avoid

In [ ]:
#   long  leg = median return of predicted Strong Buy coins
#   short leg = median return of predicted Strong Avoid coins (inverted)

weeks = sorted(test_df['year_week'].unique())
weekly_rows = []
for wk in weeks:
    m   = test_df[test_df['year_week'] == wk]
    row = {'week': wk}
    for label in LABEL_ORDER:
        b = m[m['predicted'] == label]['forward_return_7d']
        row[label]          = b.median() if len(b) else np.nan
        row[f'{label}_n']   = len(b)
        row[f'{label}_avg'] = b.mean()   if len(b) else np.nan
    long_ret  = row.get(LONG_LABEL,  np.nan)
    short_ret = row.get(SHORT_LABEL, np.nan)
    if pd.notna(long_ret) and pd.notna(short_ret):
        row['long_ret']   = long_ret
        row['short_ret']  = short_ret
        row['short_pnl']  = -short_ret
        row['net_pnl']    = long_ret - short_ret
        row['spread']     = long_ret - short_ret
    else:
        row['long_ret'] = row['short_ret'] = row['short_pnl'] = row['net_pnl'] = row['spread'] = np.nan
    weekly_rows.append(row)

weekly_perf = pd.DataFrame(weekly_rows)
valid = weekly_perf['spread'].dropna()

print(f'Test weeks          : {len(valid)}')
print(f'Avg weekly spread   : {valid.mean()*100:+.2f}%')
print(f'Median weekly spread: {valid.median()*100:+.2f}%')
print(f'Positive weeks      : {(valid > 0).mean()*100:.0f}%')
print(f'Ann. spread (52w)   : {valid.mean()*52*100:+.1f}%')

print('\nBucket summary:')
bucket_stats = []
for label in LABEL_ORDER:
    b = test_df[test_df['predicted'] == label]['forward_return_7d']
    bucket_stats.append({
        'label':    label,
        'n':        len(b),
        'avg':      b.mean()  * 100,
        'median':   b.median()* 100,
        'std':      b.std()   * 100,
        'pct_pos':  (b > 0).mean() * 100,
    })
bucket_df = pd.DataFrame(bucket_stats)
print(bucket_df.round(2).to_string(index=False))

## 4. Visualisations

In [ ]:
def cumret(s): return (1 + s.fillna(0)).cumprod()
test_start = pd.Timestamp(TRAIN_CUTOFF + '-01')
btc_w = (df[df['asset_id'] == 'bitcoin']
         .set_index('date')['close'].resample('W').last().pct_change()
         .loc[test_start:])
eth_w = (df[df['asset_id'] == 'ethereum']
         .set_index('date')['close'].resample('W').last().pct_change()
         .loc[test_start:])

n          = len(weekly_perf)
x          = range(n)
tick_step  = max(1, n // 14)
wk_labels  = weekly_perf['week'].values

sb_cum   = cumret(weekly_perf['Strong Buy']).values
sa_cum   = cumret(weekly_perf['Strong Avoid']).values
net_cum  = cumret(weekly_perf['net_pnl']).values
btc_cum  = cumret(btc_w.reset_index(drop=True).iloc[:n]).values
eth_cum  = cumret(eth_w.reset_index(drop=True).iloc[:n]).values

fig = plt.figure(figsize=(18, 26))
fig.patch.set_facecolor('white')
gs  = GridSpec(5, 2, figure=fig, hspace=0.50, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x, net_cum, color='#3C3489',    lw=2.5, label='Long/Short net (Strong Buy - Strong Avoid)')
ax1.plot(x, sb_cum,  color=COLORS['Strong Buy'],  lw=1.8, label='Long only (Strong Buy)',  linestyle='--')
ax1.plot(x, sa_cum,  color=COLORS['Strong Avoid'],lw=1.8, label='Short only (Strong Avoid inverted)', linestyle='--')
ax1.plot(x, btc_cum, color=BTC_COLOR,    lw=1.8, label='BTC',   linestyle='-.')
ax1.plot(x, eth_cum, color=ETH_COLOR,    lw=1.8, label='ETH',   linestyle=':')
ax1.axhline(1.0, color='#888780', lw=0.8, linestyle=':')
ax1.fill_between(x, net_cum, 1, where=net_cum >= 1, alpha=0.10, color='#3C3489')
ax1.fill_between(x, net_cum, 1, where=net_cum <  1, alpha=0.10, color=COLORS['Strong Avoid'])
ax1.set_xticks(range(0, n, tick_step))
ax1.set_xticklabels(wk_labels[::tick_step], rotation=45, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:.2f}x'))
ax1.set_title('Cumulative growth: long/short strategy vs BTC & ETH', fontsize=13, fontweight='500', pad=12)
ax1.legend(fontsize=9, framealpha=0)
ax1.spines[['top','right']].set_visible(False)
ax1.set_facecolor('white')

ax2 = fig.add_subplot(gs[1, :])
net_vals    = weekly_perf['net_pnl'].values
bar_cols    = [COLORS['Strong Buy'] if v > 0 else COLORS['Strong Avoid'] for v in net_vals]
ax2.bar(x, net_vals * 100, color=bar_cols, alpha=0.75, width=0.8)
ax2.axhline(0, color='#888780', lw=0.8)
ax2.axhline(valid.mean() * 100, color='#3C3489', lw=1.2, linestyle='--',
            label=f'Avg net P&L {valid.mean()*100:+.2f}%/week')
ax2.set_xticks(range(0, n, tick_step))
ax2.set_xticklabels(wk_labels[::tick_step], rotation=45, ha='right', fontsize=8)
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:+.0f}%'))
ax2.set_title('Weekly net P&L: long Strong Buy + short Strong Avoid', fontsize=13, fontweight='500', pad=12)
ax2.legend(fontsize=9, framealpha=0)
ax2.spines[['top','right']].set_visible(False)
ax2.set_facecolor('white')

ax3 = fig.add_subplot(gs[2, 0])
box_data = [test_df[test_df['predicted'] == l]['forward_return_7d'].dropna() * 100
            for l in LABEL_ORDER]
bp = ax3.boxplot(box_data, patch_artist=True, widths=0.5,
                  medianprops=dict(color='white', lw=2),
                  whiskerprops=dict(lw=1), capprops=dict(lw=1),
                  flierprops=dict(marker='.', markersize=2, alpha=0.3))
for patch, label in zip(bp['boxes'], LABEL_ORDER):
    patch.set_facecolor(COLORS[label]); patch.set_alpha(0.75)
ax3.set_xticklabels(LABEL_ORDER, fontsize=8, rotation=15)
ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:+.0f}%'))
ax3.axhline(0, color='#888780', lw=0.8, linestyle=':')
ax3.set_title('7-day return distribution per bucket', fontsize=11, fontweight='500', pad=10)
ax3.spines[['top','right']].set_visible(False)
ax3.set_facecolor('white')

ax4 = fig.add_subplot(gs[2, 1])
pct_pos = [test_df[test_df['predicted'] == l]['forward_return_7d'].gt(0).mean() * 100
           for l in LABEL_ORDER]
bars = ax4.bar(LABEL_ORDER, pct_pos, color=[COLORS[l] for l in LABEL_ORDER], alpha=0.75, width=0.5)
ax4.axhline(50, color='#888780', lw=0.8, linestyle='--', label='50% baseline')
for bar, val in zip(bars, pct_pos):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax4.set_ylim(0, 72)
ax4.set_xticklabels(LABEL_ORDER, fontsize=8, rotation=15)
ax4.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax4.set_title('Hit rate: % of weeks with positive return', fontsize=11, fontweight='500', pad=10)
ax4.legend(fontsize=8, framealpha=0)
ax4.spines[['top','right']].set_visible(False)
ax4.set_facecolor('white')

ax5 = fig.add_subplot(gs[3, 0])
imp_final = pd.Series(model.feature_importances_, index=top30).sort_values()
imp_final.tail(20).plot(kind='barh', ax=ax5, color='#378ADD', alpha=0.75)
ax5.set_title('Top 20 feature importances', fontsize=11, fontweight='500', pad=10)
ax5.set_xlabel('Importance', fontsize=9)
ax5.tick_params(axis='y', labelsize=8)
ax5.spines[['top','right']].set_visible(False)
ax5.set_facecolor('white')

ax6 = fig.add_subplot(gs[3, 1])
roll = pd.Series(net_vals).rolling(4, min_periods=1).mean() * 100
ax6.plot(x, roll, color='#3C3489', lw=2)
ax6.fill_between(x, roll, 0, where=roll >= 0, alpha=0.15, color=COLORS['Strong Buy'])
ax6.fill_between(x, roll, 0, where=roll <  0, alpha=0.15, color=COLORS['Strong Avoid'])
ax6.axhline(0, color='#888780', lw=0.8)
ax6.set_xticks(range(0, n, tick_step))
ax6.set_xticklabels(wk_labels[::tick_step], rotation=45, ha='right', fontsize=8)
ax6.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:+.1f}%'))
ax6.set_title('Rolling 4-week avg net P&L', fontsize=11, fontweight='500', pad=10)
ax6.spines[['top','right']].set_visible(False)
ax6.set_facecolor('white')

ax7 = fig.add_subplot(gs[4, :])
long_vals  = weekly_perf['long_ret'].fillna(0).values  * 100
short_vals = weekly_perf['short_pnl'].fillna(0).values * 100
ax7.bar(x, long_vals,  color=COLORS['Strong Buy'],   alpha=0.7, label='Long leg (Strong Buy)', width=0.8)
ax7.bar(x, short_vals, color=COLORS['Strong Avoid'],  alpha=0.7, label='Short leg gain (inverted Strong Avoid)',
        width=0.8, bottom=0)
ax7.plot(x, long_vals + short_vals, color='#3C3489', lw=1.5, label='Net', zorder=5)
ax7.axhline(0, color='#888780', lw=0.8)
ax7.set_xticks(range(0, n, tick_step))
ax7.set_xticklabels(wk_labels[::tick_step], rotation=45, ha='right', fontsize=8)
ax7.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:+.0f}%'))
ax7.set_title('Long vs short leg contribution per week', fontsize=11, fontweight='500', pad=10)
ax7.legend(fontsize=9, framealpha=0)
ax7.spines[['top','right']].set_visible(False)
ax7.set_facecolor('white')

plt.suptitle('Weekly Stat Arb — 5 Buckets, Long/Short 10% Strategy', fontsize=15, fontweight='500', y=1.005)
plt.savefig(DYPLOM / 'weekly_performance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved -> weekly_performance.png')

## 5. Coin-level analysis

In [ ]:
total_weeks = test_df['year_week'].nunique()

sb_counts = test_df[test_df['predicted'] == LONG_LABEL]['asset_id'].value_counts()
sa_counts = test_df[test_df['predicted'] == SHORT_LABEL]['asset_id'].value_counts()

coin_stats = pd.DataFrame({
    'long_weeks':  sb_counts,
    'short_weeks': sa_counts,
}).fillna(0).astype(int)

coin_stats['long_pct']   = coin_stats['long_weeks']  / total_weeks * 100
coin_stats['short_pct']  = coin_stats['short_weeks'] / total_weeks * 100
coin_stats['net_score']  = coin_stats['long_pct'] - coin_stats['short_pct']

avg_ret_long = (
    test_df[test_df['predicted'] == LONG_LABEL]
    .groupby('asset_id')['forward_return_7d'].mean() * 100
)
avg_ret_short = (
    test_df[test_df['predicted'] == SHORT_LABEL]
    .groupby('asset_id')['forward_return_7d'].mean() * 100
)
coin_stats['avg_ret_long']  = avg_ret_long
coin_stats['avg_ret_short'] = avg_ret_short
coin_stats = coin_stats.sort_values('net_score', ascending=False)

print(f'Total test weeks: {total_weeks}')
print('\nTop 20 coins by net score (Long % - Short %):')
print(coin_stats.head(20).round(1).to_string())
print('\nBottom 10 (most shorted):')
print(coin_stats.tail(10)[['long_weeks','short_weeks','net_score']].round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')
top20 = coin_stats.head(20)
y     = range(len(top20))

# Long vs Short selection rate
ax = axes[0]
ax.barh(y,  top20['long_pct'],  color=COLORS['Strong Buy'],   alpha=0.8, label='Long (Strong Buy) %')
ax.barh(y, -top20['short_pct'], color=COLORS['Strong Avoid'], alpha=0.8, label='Short (Strong Avoid) %')
ax.set_yticks(y)
ax.set_yticklabels(top20.index, fontsize=9)
ax.axvline(0, color='#888780', lw=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{abs(v):.0f}%'))
ax.set_title('Long vs short selection rate per coin', fontsize=11, fontweight='500', pad=10)
ax.legend(fontsize=9, framealpha=0)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

# Avg 7d return when longed
ax2 = axes[1]
ret = top20['avg_ret_long'].fillna(0)
ax2.barh(y, ret, color=[COLORS['Strong Buy'] if v >= 0 else COLORS['Strong Avoid'] for v in ret], alpha=0.8)
ax2.set_yticks(y)
ax2.set_yticklabels(top20.index, fontsize=9)
ax2.axvline(0, color='#888780', lw=0.8)
ax2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:+.1f}%'))
ax2.set_title('Avg 7d return when predicted Strong Buy', fontsize=11, fontweight='500', pad=10)
ax2.spines[['top','right']].set_visible(False)
ax2.set_facecolor('white')

plt.suptitle('Coin-level selection analysis', fontsize=13, fontweight='500')
plt.tight_layout()
plt.savefig(DYPLOM / 'weekly_coin_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved -> weekly_coin_analysis.png')

In [ ]:
# Heatmap: Strong Buy selection rate by coin x month
test_df['month'] = pd.to_datetime(test_df['date']).dt.to_period('M').astype(str)
top_coins = coin_stats.head(25).index.tolist()

heatmap_data = (
    test_df[test_df['asset_id'].isin(top_coins)]
    .assign(is_long=lambda d: (d['predicted'] == LONG_LABEL).astype(int))
    .groupby(['asset_id', 'month'])['is_long']
    .mean()
    .unstack(fill_value=0)
    * 100
)

fig, ax = plt.subplots(figsize=(18, 9))
fig.patch.set_facecolor('white')
cmap = mcolors.LinearSegmentedColormap.from_list('ls', ['#FCEBEB', '#E1F5EE', '#1D9E75'])
im   = ax.imshow(heatmap_data.values, aspect='auto', cmap=cmap, vmin=0, vmax=100)
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=9)
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        if val > 0:
            ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                    fontsize=7, color='white' if val > 55 else '#27500A')
plt.colorbar(im, ax=ax, label='% of weeks selected as Strong Buy (long)')
ax.set_title('Strong Buy selection rate by coin x month (top 25 coins)', fontsize=12, fontweight='500', pad=12)
plt.tight_layout()
plt.savefig(DYPLOM / 'weekly_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved -> weekly_heatmap.png')

In [ ]:
# Save text results
imp_final = pd.Series(model.feature_importances_, index=top30).sort_values(ascending=False)
summary = f"""
Weekly Stat Arb — 5 Buckets, Long/Short 10%
============================================
Strategy : Long top 10% (Strong Buy), Short bottom 10% (Strong Avoid)
Buckets  : Strong Avoid (<10%) | Avoid (10-30%) | Neutral (30-70%) | Buy (70-90%) | Strong Buy (>90%)

Train : {train_df['year_week'].min()} -> {train_df['year_week'].max()} ({len(train_df):,} rows)
Test  : {test_df['year_week'].min()} -> {test_df['year_week'].max()} ({len(test_df):,} rows)
Test weeks : {len(valid)}

Pass 1 OOB ({len(all_features)} features): {rf1.oob_score_*100:.2f}%
Pass 2 OOB ({TOP_N} features)            : {model.oob_score_*100:.2f}%
Test accuracy                            : {acc*100:.2f}%  (vs {baseline*100:.0f}% baseline)

Strategy Performance:
  Avg weekly net P&L  : {valid.mean()*100:+.2f}%
  Median weekly P&L   : {valid.median()*100:+.2f}%
  Ann. spread (x52)   : {valid.mean()*52*100:+.1f}%
  Positive weeks      : {(valid > 0).mean()*100:.0f}%

Bucket Summary:
{bucket_df.round(2).to_string(index=False)}

Top 30 Features:
{imp_final.to_string()}

Top 20 Coins by Net Score:
{coin_stats.head(20).round(1).to_string()}
"""
RESULTS_FILE.write_text(summary)
print(summary)